<a href="https://colab.research.google.com/github/martudiz/Funciones_python/blob/main/guiapt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def alfabeto_y_frecuencias(lista_simbolos):
    """
    Pre: 'lista_simbolos' es una lista de strings, los símbolos emitidos por una fuente.
    Post: retorna dos listas paralelas: el alfabeto (símbolos únicos, en orden de
          primera aparición) y la frecuencia absoluta de cada símbolo.
    """
    alfabeto = []
    frecuencias = []
    for simbolo in lista_simbolos:
        if simbolo in alfabeto:
            frecuencias[alfabeto.index(simbolo)] += 1
        else:
            alfabeto.append(simbolo)
            frecuencias.append(1)
    return alfabeto, frecuencias
def frecuencias_a_probabilidades(frecuencias, total):
    """
    Pre: 'frecuencias' es una lista de enteros >= 0; 'total' es la cantidad total
         de símbolos observados (> 0).
    Post: retorna la lista de probabilidades (frecuencia / total), paralela a
          'frecuencias'.
    """
    return [f / total for f in frecuencias]

In [ ]:
"""1. Codificar una función booleana en Python que reciba como parámetros: una lista con la distribución de probabilidades de una fuente, otra lista
con palabras código para la extensión de orden N y el valor de N, y verifique si el código cumple con el Primer Teorema de Shannon"""
import math


def calcular_r(palabras_codigo):
    # r = cantidad de caracteres distintos que aparecen en las palabras código
    alfabeto = []
    for palabra in palabras_codigo:
        for caracter in palabra:
            if caracter not in alfabeto:
                alfabeto.append(caracter)
    return len(alfabeto)


def calcular_entropia(probabilidades, r):
    # H_r(S) = suma de p * log_r(1/p)
    entropia = 0.0
    for p in probabilidades:
        info = math.log(1.0 / p) / math.log(r)
        entropia = entropia + p * info
    return entropia


def calcular_probs_extension(probabilidades, N):
    # Probabilidad de cada bloque de N símbolos (memoria nula):
    # producto de las probabilidades de sus símbolos
    probs_extension = [1.0]
    for paso in range(N):
        nuevas = []
        for prob_bloque in probs_extension:
            for p in probabilidades:
                nuevas.append(prob_bloque * p)
        probs_extension = nuevas
    return probs_extension


def calcular_longitud_media(probs_extension, palabras_codigo):
    # Ln = suma de P(bloque) * longitud de su palabra código
    Ln = 0.0
    for i in range(len(probs_extension)):
        Ln = Ln + probs_extension[i] * len(palabras_codigo[i])
    return Ln


def cumple_primer_teorema_shannon(probabilidades, palabras_codigo, N):
    r = calcular_r(palabras_codigo)
    entropia = calcular_entropia(probabilidades, r)
    probs_extension = calcular_probs_extension(probabilidades, N)
    Ln = calcular_longitud_media(probs_extension, palabras_codigo)
    longitud_por_simbolo = Ln / N

    # H_r(S) <= Ln/N < H_r(S) + 1/N
    tolerancia = 1e-9 #número muy chico
    cota_inferior_ok = longitud_por_simbolo >= entropia - tolerancia
    cota_superior_ok = longitud_por_simbolo < entropia + 1.0 / N

    return cota_inferior_ok and cota_superior_ok


def main():
    p = [0.9, 0.1]
    N = 2

    codigos1 = ["0", "10", "110", "111"]      # Huffman para 00, 01, 10, 11
    print("Código 1:", cumple_primer_teorema_shannon(p, codigos1, N))

    codigos2 = ["000", "001", "010", "011"]   # longitud fija, desperdicia mucho
    print("Código 2:", cumple_primer_teorema_shannon(p, codigos2, N))


main()

Código 1: True
Código 2: False


In [ ]:
"""6. Realizar una función en Python que reciba como parámetros: dos listas paralelas con la distribución de probabilidades de una fuente y su
codificación, y calcule el rendimiento y la redundancia del código.  """
import math

# --- Funciones de tu archivo integradas ---

def calcular_r(palabras_codigo):
    # r = cantidad de caracteres distintos que aparecen en las palabras código
    alfabeto = []
    for palabra in palabras_codigo:
        for caracter in palabra:
            if caracter not in alfabeto:
                alfabeto.append(caracter)
    return len(alfabeto)

def calcular_entropia(probabilidades, r):
    # H_r(S) = suma de p * log_r(1/p)
    entropia = 0.0
    for p in probabilidades:
        info = math.log(1.0 / p) / math.log(r)
        entropia = entropia + p * info
    return entropia

def calcular_longitud_media(probabilidades, palabras_codigo):
    # Ln = suma de P(símbolo) * longitud de su palabra código
    Ln = 0.0
    for i in range(len(probabilidades)):
        Ln = Ln + probabilidades[i] * len(palabras_codigo[i])
    return Ln

# --- Nuevos módulos para rendimiento y redundancia ---

def calcular_rendimiento(entropia_r, longitud_media):
    return entropia_r / longitud_media

def calcular_redundancia(rendimiento):
    return 1.0 - rendimiento

def analizar_rendimiento_y_redundancia(probabilidades, codificacion):
    """
    Orquesta las funciones modulares para devolver rendimiento y redundancia.
    """
    r = calcular_r(codificacion)

    if r < 2:
        raise ValueError("El código debe tener al menos 2 símbolos distintos.")

    entropia_r = calcular_entropia(probabilidades, r)
    L = calcular_longitud_media(probabilidades, codificacion)

    rendimiento = calcular_rendimiento(entropia_r, L)
    redundancia = calcular_redundancia(rendimiento)

    return rendimiento, redundancia

# --- Prueba de la función modularizada ---
if __name__ == "__main__":
    probabilidades_ej = [0.15, 0.25, 0.05, 0.45, 0.1]
    codigo_ej = ['(]', ']', '[]', ')', '([']

    rend, red = analizar_rendimiento_y_redundancia(probabilidades_ej, codigo_ej)
    print(f"Rendimiento: {rend:.4f} ({(rend*100):.2f}%)")
    print(f"Redundancia: {red:.4f} ({(red*100):.2f}%)")

Rendimiento: 0.7605 (76.05%)
Redundancia: 0.2395 (23.95%)


In [ ]:
"""10. Implementar una función en Python que reciba una lista de cadenas de caracteres que representa una codificación binaria y devuelva:
la distancia de Hamming, la cantidad de errores que se pueden detectar y la cantidad de errores que se pueden corregir. """
def distancia_hamming(palabra1, palabra2):
    """
    Calcula la distancia de Hamming entre dos cadenas de igual longitud
    iterando por índice.
    """
    distancia = 0
    for i in range(len(palabra1)):
        if palabra1[i] != palabra2[i]:
            distancia += 1
    return distancia

def analizar_capacidades_codigo(codificacion):
    """
    Recibe una lista de palabras código (cadenas binarias).
    Retorna la distancia de Hamming mínima, errores detectables y errores corregibles.
    """
    # 1. Calcular la distancia de Hamming mínima del código (d_min)
    d_min = float('inf') #valor infinito centinela
    cantidad_palabras = len(codificacion)

    for i in range(cantidad_palabras):
        for j in range(i + 1, cantidad_palabras):
            dist = distancia_hamming(codificacion[i], codificacion[j])
            if dist < d_min:
                d_min = dist

    # Si solo hay una palabra, la distancia no está definida de forma estándar
    if d_min == float('inf'):
        d_min = 0

    # 2. Calcular errores detectables (d - 1)
    errores_detectables = max(0, d_min - 1)

    # 3. Calcular errores corregibles floor((d - 1) / 2)
    errores_corregibles = max(0, (d_min - 1) // 2)

    return d_min, errores_detectables, errores_corregibles

# --- Prueba de la función ---
if __name__ == "__main__":
    codigo_binario = ["00000", "01101", "10110", "11011"]
    d, detectables, corregibles = analizar_capacidades_codigo(codigo_binario)

    print(f"Distancia de Hamming del código: {d}")
    print(f"Errores que se pueden detectar: {detectables}")
    print(f"Errores que se pueden corregir: {corregibles}")

Distancia de Hamming del código: 3
Errores que se pueden detectar: 2
Errores que se pueden corregir: 1


In [ ]:
"""12. Desarrollar funciones en Python que resuelvan lo siguiente:
a. Dado un carácter, devolver un byte que represente su código ASCII (7 bits) y utilice el bit menos significativo para almacenar
   la paridad del código.
b. Dado un byte que se obtuvo como resultado de la función anterior, verificar si es correcto o tiene errores. """
def codificar_con_paridad(caracter):
    """
    Inciso A: Recibe un carácter, obtiene su valor ASCII de 7 bits,
    lo desplaza hacia la izquierda para dejar el bit menos significativo (LSB) libre,
    y asigna ahí el bit de paridad (paridad par).
    """
    # Obtener el código ASCII asegurando que sea de 7 bits (0-127)
    ascii_val = ord(caracter) & 0x7F

    # Contar cuántos '1' tiene la representación binaria del código ASCII
    cantidad_unos = bin(ascii_val).count('1')

    # Paridad par: el bit será 1 si la cantidad de unos actual es impar
    bit_paridad = cantidad_unos % 2

    # Desplazar ASCII 1 bit a la izquierda y aplicar el bit de paridad en el LSB
    byte_codificado = (ascii_val << 1) | bit_paridad

    return byte_codificado

def verificar_byte(byte):
    """
    Inciso B: Verifica si el byte recibido es correcto asumiendo esquema de paridad par.
    """
    # Contar la cantidad total de '1' en el byte recibido
    cantidad_unos = bin(byte).count('1')

    # Si la cantidad total de unos es par, no se detectan errores de bit individual
    if cantidad_unos % 2 == 0:
        # Retornamos True (correcto) y el carácter original (descartando el LSB)
        return True, chr(byte >> 1)
    else:
        # Retornamos False (error detectado) y None
        return False, None

# --- Prueba de las funciones ---
if __name__ == "__main__":
    letra = 'A'

    # 1. Codificación
    byte_generado = codificar_con_paridad(letra)
    print(f"Carácter original: '{letra}' (ASCII: {ord(letra)}, Binario 7-bits: {bin(ord(letra))[2:].zfill(7)})")
    print(f"Byte codificado (Binario 8-bits): {bin(byte_generado)[2:].zfill(8)}")

    # 2. Verificación sin errores
    es_valido, char_decodificado = verificar_byte(byte_generado)
    print(f"Verificación byte original: {'Correcto' if es_valido else 'Con errores'} -> Carácter: {char_decodificado}")

    # 3. Simulación de un error (modificando un bit al azar con un XOR)
    byte_con_error = byte_generado ^ 0b00000100
    es_valido_err, _ = verificar_byte(byte_con_error)
    print(f"Byte con ruido (Binario 8-bits):  {bin(byte_con_error)[2:].zfill(8)}")
    print(f"Verificación byte modificado: {'Correcto' if es_valido_err else 'Con errores detectados por paridad'}")

Carácter original: 'A' (ASCII: 65, Binario 7-bits: 1000001)
Byte codificado (Binario 8-bits): 10000010
Verificación byte original: Correcto -> Carácter: A
Byte con ruido (Binario 8-bits):  10000110
Verificación byte modificado: Con errores detectados por paridad


In [ ]:
""" 14. Codificar funciones en Python que resuelvan lo siguiente:
a. Dada una cadena de caracteres, generar una secuencia de bytes (bytearray) que contenga su representación con código ASCII y
   sus bits de paridad vertical, longitudinal y cruzada.
b. Dada una secuencia de bytes que se obtuvo como resultado de la función anterior, devolver el mensaje original o una cadena
de caracteres vacía si no se pueden corregir los errores.

Sugerencia: generar una matriz de bits para realizar las operaciones, transformando la secuencia de bytes en una lista de cadenas
de caracteres binarias y, luego, cada cadena de caracteres en una lista de números enteros que representen los bits.  """
def generar_matriz_bits(cadena):
    """
    Transforma la cadena en una lista de listas de enteros (matriz de bits).
    Cada carácter se representa en 7 bits ASCII y se le agrega un bit de
    paridad vertical (VRC) par al final de la fila.
    """
    matriz = []
    for i in range(len(cadena)):
        ascii_val = ord(cadena[i])
        fila_bits = []
        # Extraer 7 bits de mayor a menor peso
        for j in range(6, -1, -1):
            fila_bits.append((ascii_val >> j) & 1)

        # Calcular paridad vertical (VRC par) iterando por índice
        suma_bits = 0
        for j in range(len(fila_bits)):
            suma_bits += fila_bits[j]

        bit_paridad = suma_bits % 2
        fila_bits.append(bit_paridad)
        matriz.append(fila_bits)

    return matriz

def calcular_lrc(matriz):
    """
    Calcula el byte de paridad longitudinal (LRC) iterando por las columnas
    de la matriz de bits. Retorna una lista de 8 bits.
    """
    lrc = [0] * 8
    for col in range(8):
        suma_columna = 0
        for fila in range(len(matriz)):
            suma_columna += matriz[fila][col]
        lrc[col] = suma_columna % 2
    return lrc

def codificar_mensaje(cadena):
    """
    Inciso A: Genera un bytearray con la representación ASCII,
    paridad vertical (VRC) y paridad longitudinal cruzada (LRC).
    """
    matriz = generar_matriz_bits(cadena)

    # Calcular y agregar la fila LRC al final de la matriz
    fila_lrc = calcular_lrc(matriz)
    matriz.append(fila_lrc)

    # Convertir la matriz final a una secuencia de bytes
    secuencia_bytes = bytearray()
    for i in range(len(matriz)):
        byte_val = 0
        for j in range(8):
            byte_val = (byte_val << 1) | matriz[i][j]
        secuencia_bytes.append(byte_val)

    return secuencia_bytes

def bytes_a_matriz(secuencia_bytes):
    """
    Convierte un bytearray nuevamente a una matriz de bits (lista de listas).
    """
    matriz = []
    for i in range(len(secuencia_bytes)):
        byte_val = secuencia_bytes[i]
        fila_bits = []
        for j in range(7, -1, -1):
            fila_bits.append((byte_val >> j) & 1)
        matriz.append(fila_bits)
    return matriz

def decodificar_y_corregir(secuencia_bytes):
    """
    Inciso B: Recibe un bytearray, verifica la paridad cruzada, corrige
    errores simples si es posible, y retorna el mensaje original.
    Si hay errores incorregibles, retorna una cadena vacía.
    """
    matriz = bytes_a_matriz(secuencia_bytes)
    print(matriz)

    # Identificar errores en las filas (VRC) omitiendo la última fila (LRC original)
    error_fila = -1
    conteo_errores_filas = 0
    filas_datos = len(matriz) - 1
    #corroborar antes cruzado y errores en paridad vertical y horizontal
    for i in range(filas_datos):
        suma_fila = 0
        for j in range(8):
            suma_fila += matriz[i][j]
        if suma_fila % 2 != 0:
            error_fila = i
            conteo_errores_filas += 1

    # Identificar errores en las columnas (LRC) evaluando la matriz completa
    error_columna = -1
    conteo_errores_cols = 0
    for j in range(8):
        suma_columna = 0
        for i in range(len(matriz)):
            suma_columna += matriz[i][j]
        if suma_columna % 2 != 0:
            error_columna = j
            conteo_errores_cols += 1

    # Lógica de corrección de paridad cruzada
    if conteo_errores_filas == 1 and conteo_errores_cols == 1:
        # Error simple detectado: se invierte el bit en la intersección
        matriz[error_fila][error_columna] ^= 1
    elif conteo_errores_filas > 1 or conteo_errores_cols > 1:
        # Múltiples errores irrecuperables
        return ""

    # Reconstrucción del mensaje decodificado
    mensaje = ""
    for i in range(filas_datos):
        ascii_val = 0
        # Leer solo los primeros 7 bits, descartando el 8vo (VRC)
        for j in range(7):
            ascii_val = (ascii_val << 1) | matriz[i][j]
        mensaje += chr(ascii_val)

    return mensaje

# --- Prueba de las funciones ---
if __name__ == "__main__":
    texto_original = "HOLA"

    # 1. Codificación
    secuencia_codificada = codificar_mensaje(texto_original)
    print("Secuencia generada (bytes):", list(secuencia_codificada))

    # 2. Decodificación sin errores
    print("Decodificación limpia:", decodificar_y_corregir(secuencia_codificada))

    # 3. Simulación de 1 error (alteramos un bit del primer carácter)
    secuencia_con_error = bytearray(secuencia_codificada)
    secuencia_con_error[0] ^= 0b00000100
    print("Decodificación con 1 error corregido:", decodificar_y_corregir(secuencia_con_error))

    # 4. Simulación de errores múltiples (irrecuperables)
    secuencia_mult_error = bytearray(secuencia_codificada)
    secuencia_mult_error[0] ^= 0b00000100
    secuencia_mult_error[1] ^= 0b00000010
    print("Decodificación con múltiples errores:", repr(decodificar_y_corregir(secuencia_mult_error)))

Secuencia generada (bytes): [144, 159, 153, 130, 20]
[[1, 0, 0, 1, 0, 0, 0, 0], [1, 0, 0, 1, 1, 1, 1, 1], [1, 0, 0, 1, 1, 0, 0, 1], [1, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 0, 1, 0, 0]]
Decodificación limpia: HOLA
[[1, 0, 0, 1, 0, 1, 0, 0], [1, 0, 0, 1, 1, 1, 1, 1], [1, 0, 0, 1, 1, 0, 0, 1], [1, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 0, 1, 0, 0]]
Decodificación con 1 error corregido: HOLA
[[1, 0, 0, 1, 0, 1, 0, 0], [1, 0, 0, 1, 1, 1, 0, 1], [1, 0, 0, 1, 1, 0, 0, 1], [1, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 0, 1, 0, 0]]
Decodificación con múltiples errores: ''


In [ ]:
"""3. Desarrollar dos funciones en Python que reciban como parámetro una lista con la distribución de probabilidades de una fuente de información
y generen una lista de cadenas de caracteres con codificaciones binarias de Huffman y de Shannon-Fano. """
def crea_huffman(probs):
  cant = len(probs)
  codigos = []
  probs_grupos = []
  simbolos_grupos = []
  for i in range(cant):
    codigos.append("")
    probs_grupos.append(probs[i])
    simbolos_grupos.append([i])

  while len(probs_grupos) > 1:
    # índice del menor
    a = 0
    for k in range(len(probs_grupos)):
      if probs_grupos[k] < probs_grupos[a]:
        a = k

    # índice del segundo menor (salteando a)
    b = -1
    for k in range(len(probs_grupos)):
      if k != a:
        if b == -1 or probs_grupos[k] < probs_grupos[b]:
          b = k

    for s in simbolos_grupos[a]:
      codigos[s] = "0" + codigos[s]
    for s in simbolos_grupos[b]:
      codigos[s] = "1" + codigos[s]

    nueva_prob = probs_grupos[a] + probs_grupos[b]
    nuevos_simbolos = simbolos_grupos[a] + simbolos_grupos[b]

    # borrar primero el índice mayor
    if a > b:
      probs_grupos.pop(a)
      simbolos_grupos.pop(a)
      probs_grupos.pop(b)
      simbolos_grupos.pop(b)
    else:
      probs_grupos.pop(b)
      simbolos_grupos.pop(b)
      probs_grupos.pop(a)
      simbolos_grupos.pop(a)

    probs_grupos.append(nueva_prob)
    simbolos_grupos.append(nuevos_simbolos)

  return codigos


def ordena_indices(probs):
  indices = []
  for i in range(len(probs)):
    indices.append(i)
  # burbuja de mayor a menor, estable
  for i in range(len(indices) - 1):
    for k in range(len(indices) - 1 - i):
      if probs[indices[k]] < probs[indices[k + 1]]:
        auxiliar = indices[k]
        indices[k] = indices[k + 1]
        indices[k + 1] = auxiliar
  return indices


def divide(indices, probs, codigos):
  if len(indices) == 1:
    return

  total = 0
  for i in indices:
    total = total + probs[i]

  corte = 1
  mejor_dif = total + 1
  izq = 0
  for k in range(1, len(indices)):
    izq = izq + probs[indices[k - 1]]
    diferencia = abs(total - 2 * izq)
    if diferencia < mejor_dif:
      mejor_dif = diferencia
      corte = k

  izquierda = indices[:corte]
  derecha = indices[corte:]

  for i in izquierda:
    codigos[i] = codigos[i] + "0"
  for i in derecha:
    codigos[i] = codigos[i] + "1"

  divide(izquierda, probs, codigos)
  divide(derecha, probs, codigos)


def crea_shannon_fano(probs):
  codigos = []
  for i in range(len(probs)):
    codigos.append("")
  indices = ordena_indices(probs)
  divide(indices, probs, codigos)
  return codigos


def main():
  fuentes = [
    [0.2, 0.2, 0.3, 0.3],
    [0.4, 0.25, 0.25, 0.1],
    [0.385, 0.154, 0.128, 0.154, 0.179]
  ]

  for probs in fuentes:
    print("Probabilidades:", probs)
    print("  Huffman:", crea_huffman(probs))
    print("  Shannon-Fano:", crea_shannon_fano(probs))

main()

Probabilidades: [0.2, 0.2, 0.3, 0.3]
  Huffman: ['00', '01', '10', '11']
  Shannon-Fano: ['10', '11', '00', '01']
Probabilidades: [0.4, 0.25, 0.25, 0.1]
  Huffman: ['0', '111', '10', '110']
  Shannon-Fano: ['0', '10', '110', '111']
Probabilidades: [0.385, 0.154, 0.128, 0.154, 0.179]
  Huffman: ['0', '101', '100', '110', '111']
  Shannon-Fano: ['00', '10', '111', '110', '01']


In [ ]:
"""7. Implementar funciones en Python que reciban como parámetros: una cadena de caracteres que contenga un alfabeto fuente y una lista de
cadenas de caracteres que almacena una codificación en el alfabeto binario, y resuelvan lo siguiente:
a. Dada una cadena de caracteres con un mensaje escrito en el alfabeto fuente, devolver una secuencia de bytes (bytearray)
que contenga el mensaje codificado.
b. Dada una secuencia de bytes, decodificar y retornar el mensaje original. """
def codificar(alfabeto, codigos, mensaje):
  bits = ""
  for caracter in mensaje:
    posicion = alfabeto.index(caracter)
    bits = bits + codigos[posicion]

  relleno = (8 - len(bits) % 8) % 8
  for i in range(relleno):
    bits = bits + "0"

  resultado = bytearray()
  resultado.append(relleno)
  for k in range(0, len(bits), 8):
    grupo = bits[k:k + 8]
    resultado.append(int(grupo, 2))
  return resultado


def decodificar(alfabeto, codigos, secuencia):
  relleno = secuencia[0]
  bits = ""
  for k in range(1, len(secuencia)):
    bits = bits + format(secuencia[k], "08b")
  bits = bits[:len(bits) - relleno]

  mensaje = ""
  acumulado = ""
  for bit in bits:
    acumulado = acumulado + bit
    if acumulado in codigos:
      posicion = codigos.index(acumulado)
      mensaje = mensaje + alfabeto[posicion]
      acumulado = ""
  return mensaje


def main():
  alfabeto = "ABCD"
  probs = [0.26, 0.4, 0.24, 0.1]
  codigos = crea_huffman(probs)
  print("Códigos:", codigos)

  mensaje = "ABCDABCBDCBA"
  secuencia = codificar(alfabeto, codigos, mensaje)
  print("Bytes:", list(secuencia))

  recuperado = decodificar(alfabeto, codigos, secuencia)
  print("Decodificado:", recuperado)
  print("Coincide:", recuperado == mensaje)

main()

Códigos: ['10', '0', '111', '110']
Bytes: [7, 159, 78, 221, 0]
Decodificado: ABCDABCBDCBA
Coincide: True


In [ ]:
"""8. Codificar una función en Python que reciba como parámetros: una cadena de caracteres con un mensaje y una secuencia de bytes (bytearray)
con ese mensaje codificado y calcule la tasa de compresión. """
def tasa_compresion(mensaje, secuencia):
  tamano_original = len(mensaje)
  tamano_comprimido = len(secuencia)
  return tamano_original / tamano_comprimido


def main():
  alfabeto = "ABCD"
  probs = [0.26, 0.4, 0.24, 0.1]
  codigos = crea_huffman(probs)

  mensaje = "ABCDABCBDCBA"
  secuencia = codificar(alfabeto, codigos, mensaje)
  print("Mensaje 1:")
  print("  Tamaño original:", len(mensaje))
  print("  Tamaño comprimido:", len(secuencia))
  print("  Tasa de compresión:", tasa_compresion(mensaje, secuencia))

  mensaje = "ABCDABCBDCBAAABBBCBCBABADBCBABCBDBCCCAAABB"
  secuencia = codificar(alfabeto, codigos, mensaje)
  print("Mensaje 2 (ejercicio 2a):")
  print("  Tamaño original:", len(mensaje))
  print("  Tamaño comprimido:", len(secuencia))
  print("  Tasa de compresión:", tasa_compresion(mensaje, secuencia))

main()

Mensaje 1:
  Tamaño original: 12
  Tamaño comprimido: 5
  Tasa de compresión: 2.4
Mensaje 2 (ejercicio 2a):
  Tamaño original: 42
  Tamaño comprimido: 12
  Tasa de compresión: 3.5


In [ ]:
"""11. Realizar una función en Python que reciba como parámetro una cadena de caracteres con un mensaje y devuelva una secuencia de bytes
(bytearray) que contenga el mensaje comprimido con RLC, utilizando un byte para almacenar la representación en código ASCII del carácter y
otro byte para el número. """
def comprimir_rlc(mensaje):
  resultado = bytearray()
  caracter_actual = mensaje[0]
  contador = 1

  for k in range(1, len(mensaje)):
    if mensaje[k] == caracter_actual:
      contador = contador + 1
    else:
      resultado.append(ord(caracter_actual))
      resultado.append(contador)
      caracter_actual = mensaje[k]
      contador = 1

  resultado.append(ord(caracter_actual))
  resultado.append(contador)
  return resultado


def main():
  mensajes = ["XXXYZZZZ", "AAAABBBCCDAA", "UUOOOOAAAIEUUUU"]

  for mensaje in mensajes:
    secuencia = comprimir_rlc(mensaje)
    print("Mensaje:", mensaje)
    print("  Bytes:", list(secuencia))

main()

Mensaje: XXXYZZZZ
  Bytes: [88, 3, 89, 1, 90, 4]
Mensaje: AAAABBBCCDAA
  Bytes: [65, 4, 66, 3, 67, 2, 68, 1, 65, 2]
Mensaje: UUOOOOAAAIEUUUU
  Bytes: [85, 2, 79, 4, 65, 3, 73, 1, 69, 1, 85, 4]


In [ ]:
"""2. Codificar una función en Python que reciba como parámetros dos cadenas de caracteres que contengan secuencias de entrada y de salida
de un canal y retorne la matriz que representa dicho canal."""
def calcular_alfabeto(cadena):
  alfabeto = []
  for caracter in cadena:
    if caracter not in alfabeto:
      alfabeto.append(caracter)
  return len(alfabeto),alfabeto
def normalizar_filas(matriz):
  for i in range(len(matriz)):
    total_fila = 0
    for j in range(len(matriz[i])):
      total_fila = total_fila + matriz[i][j]
    for j in range(len(matriz[i])):
      matriz[i][j] = matriz[i][j] / total_fila
  return matriz
def genero_matriz(cadena_entrada,cadena_salida):
  cant_filas,alfabeto_entrada = calcular_alfabeto(cadena_entrada)
  cant_columnas,alfabeto_salida = calcular_alfabeto(cadena_salida)
  matriz = []
  for i in range(cant_filas):
    fila = []
    for j in range(cant_columnas):
      fila.append(0)
    matriz.append(fila)
  for k in range(len(cadena_entrada)):
    fila = alfabeto_entrada.index(cadena_entrada[k])
    columna = alfabeto_salida.index(cadena_salida[k])
    matriz[fila][columna] = matriz[fila][columna] + 1
  matriz = normalizar_filas(matriz)
  return matriz
def main():
  matriz = genero_matriz("AABBBCCABC", "AABCBCCBBC")
  for fila in matriz:
    print(fila)

main()



[0.6666666666666666, 0.3333333333333333, 0.0]
[0.0, 0.75, 0.25]
[0.0, 0.0, 1.0]


In [ ]:
"""7. Realizar funciones en Python que reciban como parámetros: una lista con las probabilidades a priori y la matriz de probabilidades
condicionales del canal, y devuelvan:
a. Una lista con las probabilidades de los símbolos de salida
b. Una matriz con las probabilidades a posteriori del canal
c. Una matriz con las probabilidades de los eventos simultáneos """

def matriz_ceros(matriz):
  cant_filas = len(matriz)
  cant_columnas = len(matriz[0])
  matriz_salida = []
  for i in range(cant_filas):
    fila = []
    for j in range(cant_columnas):
      fila.append(0)
    matriz_salida.append(fila)
  return matriz_salida

def crea_matriz_eventos_simultaneos(matriz_canal,prob_priori):
  matriz_eventos_simultaneos = matriz_ceros(matriz_canal)
  cant_filas = len(matriz_canal)
  cant_columnas = len(matriz_canal[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      matriz_eventos_simultaneos[i][j] = matriz_canal[i][j] * prob_priori[i]
  return matriz_eventos_simultaneos

def crea_prob_simbolos_salida(matriz_eventos_simultaneos):
  cant_columnas=len(matriz_eventos_simultaneos[0])
  cant_filas=len(matriz_eventos_simultaneos)
  prob_simbolos_salida = []
  for j in range(cant_columnas):
    prob_simbolos_salida.append(0)
  for j in range(cant_columnas):
    for i in range(cant_filas):
      prob_simbolos_salida[j]+=matriz_eventos_simultaneos[i][j]
  return prob_simbolos_salida
def crea_matriz_posteriori(matriz_eventos_simultaneos,prob_simbolos_salida):
  matriz_posteriori = matriz_ceros(matriz_eventos_simultaneos)
  cant_filas = len(matriz_eventos_simultaneos)
  cant_columnas = len(matriz_eventos_simultaneos[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      matriz_posteriori[i][j] = matriz_eventos_simultaneos[i][j] / prob_simbolos_salida[j]
  return matriz_posteriori

def main():
  prob_priori = [0.5, 0.5]
  matriz_canal = [[0.5, 0.5, 0.0], [0.5, 0.0, 0.5]]

  simultaneos = crea_matriz_eventos_simultaneos(matriz_canal, prob_priori)
  prob_salida = crea_prob_simbolos_salida(simultaneos)
  posteriori = crea_matriz_posteriori(simultaneos, prob_salida)

  print("Eventos simultáneos:")
  for fila in simultaneos:
    print(fila)

  print("Probabilidades de salida:")
  print(prob_salida)

  print("A posteriori:")
  for fila in posteriori:
    print(fila)

main()

Eventos simultáneos:
[0.25, 0.25, 0.0]
[0.25, 0.0, 0.25]
Probabilidades de salida:
[0.5, 0.25, 0.25]
A posteriori:
[0.5, 1.0, 0.0]
[0.5, 0.0, 1.0]


In [ ]:
"""11. Desarrollar una función en Python que reciba como parámetros: una lista con las probabilidades a priori y la matriz de probabilidades
condicionales del canal, y retorne una lista con las entropías a posteriori."""
import math
def matriz_ceros(matriz):
  cant_filas = len(matriz)
  cant_columnas = len(matriz[0])
  matriz_salida = []
  for i in range(cant_filas):
    fila = []
    for j in range(cant_columnas):
      fila.append(0)
    matriz_salida.append(fila)
  return matriz_salida

def crea_matriz_eventos_simultaneos(matriz_canal,prob_priori):
  matriz_eventos_simultaneos = matriz_ceros(matriz_canal)
  cant_filas = len(matriz_canal)
  cant_columnas = len(matriz_canal[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      matriz_eventos_simultaneos[i][j] = matriz_canal[i][j] * prob_priori[i]
  return matriz_eventos_simultaneos

def crea_prob_simbolos_salida(matriz_eventos_simultaneos):
  cant_columnas=len(matriz_eventos_simultaneos[0])
  cant_filas=len(matriz_eventos_simultaneos)
  prob_simbolos_salida = []
  for j in range(cant_columnas):
    prob_simbolos_salida.append(0)
  for j in range(cant_columnas):
    for i in range(cant_filas):
      prob_simbolos_salida[j]+=matriz_eventos_simultaneos[i][j]
  return prob_simbolos_salida

def crea_matriz_posteriori(matriz_eventos_simultaneos,prob_simbolos_salida):
  matriz_posteriori = matriz_ceros(matriz_eventos_simultaneos)
  cant_filas = len(matriz_eventos_simultaneos)
  cant_columnas = len(matriz_eventos_simultaneos[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      matriz_posteriori[i][j] = matriz_eventos_simultaneos[i][j] / prob_simbolos_salida[j]
  return matriz_posteriori

def calcular_entropia_posteriori(matriz_canal,prob_priori):
  simultaneos = crea_matriz_eventos_simultaneos(matriz_canal, prob_priori)
  prob_salida = crea_prob_simbolos_salida(simultaneos)
  posteriori = crea_matriz_posteriori(simultaneos, prob_salida)
  cant_columnas=len(posteriori[0])
  entropia_posteriori=[]
  for j in range(cant_columnas):
    entropia_posteriori.append(0)
  for j in range(cant_columnas):
    for i in range(len(posteriori)):
      if posteriori[i][j] > 0:
        entropia_posteriori[j] = entropia_posteriori[j] + posteriori[i][j] * math.log2(1 / posteriori[i][j])
  return entropia_posteriori
def main():
  prob_priori = [0.5, 0.5]

  matriz_canal = [[1.0, 0.0], [0.5, 0.5]]
  entropias = calcular_entropia_posteriori(matriz_canal, prob_priori)
  print("Ejemplo 1:")
  print(entropias)

  matriz_canal = [[0.5, 0.5, 0.0], [0.5, 0.0, 0.5]]
  entropias = calcular_entropia_posteriori(matriz_canal, prob_priori)
  print("Ejemplo 2:")
  print(entropias)

main()


Ejemplo 1:
[0.9182958340544893, 0.0]
Ejemplo 2:
[1.0, 0.0, 0.0]


In [ ]:
"""15. Implementar funciones en Python que reciban como parámetros: una lista con las probabilidades a priori y la matriz de
probabilidades condicionales del canal, y calculen por definición (es decir, a partir de las probabilidades, no de sus relaciones):
a. La equivocación o ruido
b. La pérdida
c. La entropía afín
d. La información mutua """
import math
def matriz_ceros(matriz):
  cant_filas = len(matriz)
  cant_columnas = len(matriz[0])
  matriz_salida = []
  for i in range(cant_filas):
    fila = []
    for j in range(cant_columnas):
      fila.append(0)
    matriz_salida.append(fila)
  return matriz_salida

def crea_matriz_eventos_simultaneos(matriz_canal,prob_priori):
  matriz_eventos_simultaneos = matriz_ceros(matriz_canal)
  cant_filas = len(matriz_canal)
  cant_columnas = len(matriz_canal[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      matriz_eventos_simultaneos[i][j] = matriz_canal[i][j] * prob_priori[i]
  return matriz_eventos_simultaneos

def crea_matriz_posteriori(matriz_eventos_simultaneos,prob_simbolos_salida):
  matriz_posteriori = matriz_ceros(matriz_eventos_simultaneos)
  cant_filas = len(matriz_eventos_simultaneos)
  cant_columnas = len(matriz_eventos_simultaneos[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      matriz_posteriori[i][j] = matriz_eventos_simultaneos[i][j] / prob_simbolos_salida[j]
  return matriz_posteriori

def crea_prob_simbolos_salida(matriz_eventos_simultaneos):
  cant_columnas=len(matriz_eventos_simultaneos[0])
  cant_filas=len(matriz_eventos_simultaneos)
  prob_simbolos_salida = []
  for j in range(cant_columnas):
    prob_simbolos_salida.append(0)
  for j in range(cant_columnas):
    for i in range(cant_filas):
      prob_simbolos_salida[j]+=matriz_eventos_simultaneos[i][j]
  return prob_simbolos_salida
def calcula_valores(prob_priori, matriz_canal):
  simultaneos = crea_matriz_eventos_simultaneos(matriz_canal, prob_priori)
  prob_salida = crea_prob_simbolos_salida(simultaneos)
  posteriori = crea_matriz_posteriori(simultaneos, prob_salida)
  return posteriori, prob_salida, simultaneos
def calcular_equivocacion(prob_priori, matriz_canal):
  posteriori, prob_salida, simultaneos = calcula_valores(prob_priori, matriz_canal)
  acumulado=0
  cant_filas = len(posteriori)
  cant_columnas = len(posteriori[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      if simultaneos[i][j] > 0:
        acumulado = acumulado + simultaneos[i][j] * math.log2(1 / posteriori[i][j])
  return acumulado
def calcular_perdida(prob_priori, matriz_canal):
  posteriori, prob_salida, simultaneos = calcula_valores(prob_priori, matriz_canal)
  acumulado=0
  cant_filas = len(posteriori)
  cant_columnas = len(posteriori[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      if simultaneos[i][j] > 0:
        acumulado = acumulado + simultaneos[i][j] * math.log2(1 / matriz_canal[i][j])
  return acumulado
def calcular_entropia_afin(prob_priori, matriz_canal):
  posteriori, prob_salida, simultaneos = calcula_valores(prob_priori, matriz_canal)
  acumulado=0
  cant_filas = len(posteriori)
  cant_columnas = len(posteriori[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      if simultaneos[i][j] > 0:
        acumulado = acumulado + simultaneos[i][j] * math.log2(1 / simultaneos[i][j])
  return acumulado
def calcular_informacion_mutua(prob_priori, matriz_canal):
  posteriori, prob_salida, simultaneos = calcula_valores(prob_priori, matriz_canal)
  acumulado=0
  cant_filas = len(posteriori)
  cant_columnas = len(posteriori[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      if simultaneos[i][j] > 0:
        acumulado = acumulado + simultaneos[i][j] * math.log2(simultaneos[i][j] / (prob_priori[i] * prob_salida[j]))
  return acumulado

def main():
  prob_priori = [0.5, 0.5]

  matriz_canal = [[1.0, 0.0], [0.5, 0.5]]
  print("Ejemplo 1:")
  print("Equivocación H(A|B):", calcular_equivocacion(prob_priori, matriz_canal))
  print("Pérdida H(B|A):", calcular_perdida(prob_priori, matriz_canal))
  print("Entropía afín H(A,B):", calcular_entropia_afin(prob_priori, matriz_canal))
  print("Información mutua I(A;B):", calcular_informacion_mutua(prob_priori, matriz_canal))

  matriz_canal = [[0.5, 0.5, 0.0], [0.5, 0.0, 0.5]]
  print("Ejemplo 2:")
  print("Equivocación H(A|B):", calcular_equivocacion(prob_priori, matriz_canal))
  print("Pérdida H(B|A):", calcular_perdida(prob_priori, matriz_canal))
  print("Entropía afín H(A,B):", calcular_entropia_afin(prob_priori, matriz_canal))
  print("Información mutua I(A;B):", calcular_informacion_mutua(prob_priori, matriz_canal))

main()

Ejemplo 1:
Equivocación H(A|B): 0.6887218755408671
Pérdida H(B|A): 0.5
Entropía afín H(A,B): 1.5
Información mutua I(A;B): 0.31127812445913283
Ejemplo 2:
Equivocación H(A|B): 0.5
Pérdida H(B|A): 1.0
Entropía afín H(A,B): 2.0
Información mutua I(A;B): 0.5


In [ ]:
"""18. Realizar funciones booleanas en Python que reciban como parámetro la matriz de un canal y verifiquen si se trata de:
a. Un canal sin ruido
b. Un canal determinante """
def canal_sin_ruido(matriz_canal):
  cant_filas = len(matriz_canal)
  cant_columnas = len(matriz_canal[0])
  for j in range(cant_columnas):
    cont=0
    for i in range(cant_filas):
      if matriz_canal[i][j]!=0:
        cont+=1
    if cont!=1:
      return False
  return True
def canal_determinante(matriz_canal):
  cant_filas = len(matriz_canal)
  cant_columnas = len(matriz_canal[0])
  for i in range(cant_filas):
    cont=0
    for j in range(cant_columnas):
      if matriz_canal[i][j]!=0:
        cont+=1
    if cont!=1:
      return False
  return True
def main():
  matrices = [
    [[1, 0], [0, 1]],
    [[0.5, 0.5, 0], [0, 0, 1]],
    [[1, 0], [0, 1], [0, 1]],
    [[0.5, 0.5], [0.5, 0.5]]
  ]

  for matriz in matrices:
    print("Matriz:", matriz)
    print("  Sin ruido:", canal_sin_ruido(matriz))
    print("  Determinante:", canal_determinante(matriz))

main()

Matriz: [[1, 0], [0, 1]]
  Sin ruido: True
  Determinante: True
Matriz: [[0.5, 0.5, 0], [0, 0, 1]]
  Sin ruido: True
  Determinante: False
Matriz: [[1, 0], [0, 1], [0, 1]]
  Sin ruido: False
  Determinante: True
Matriz: [[0.5, 0.5], [0.5, 0.5]]
  Sin ruido: False
  Determinante: False


In [ ]:
"""20. Implementar una función en Python que reciba como parámetros las matrices de dos canales y genere la matriz del canal compuesto por los
canales en serie. """
def matriz_ceros(matriz, filas, columnas):
  for i in range(filas):
    fila = []
    for j in range(columnas):
      fila.append(0)
    matriz.append(fila)
  return matriz
def crea_matriz_compuesta(matriz_canal1, matriz_canal2):
  cant_filas=len(matriz_canal1)
  cant_columnas=len(matriz_canal2[0])
  dimension=len(matriz_canal2)
  matriz_compuesta = []
  matriz_compuesta = matriz_ceros(matriz_compuesta, cant_filas, cant_columnas)
  for i in range(cant_filas):
    for k in range(cant_columnas):
      for j in range(dimension):
        matriz_compuesta[i][k] = matriz_compuesta[i][k] + matriz_canal1[i][j] * matriz_canal2[j][k]
  return matriz_compuesta
def main():
  canal1 = [[1.0, 0.0], [0.5, 0.5]]
  canal2 = [[0.5, 0.5], [0.0, 1.0]]
  print("Ejemplo 1:")
  for fila in crea_matriz_compuesta(canal1, canal2):
    print(fila)

  canal1 = [[0.5, 0.5, 0.0], [0.0, 0.5, 0.5]]
  canal2 = [[1.0, 0.0], [0.5, 0.5], [0.0, 1.0]]
  print("Ejemplo 2:")
  for fila in crea_matriz_compuesta(canal1, canal2):
    print(fila)

main()

Ejemplo 1:
[0.5, 0.5]
[0.25, 0.75]
Ejemplo 2:
[0.75, 0.25]
[0.25, 0.75]


In [ ]:
"""22. Desarrollar funciones en Python que reciban como parámetros: la matriz de un canal y los índices de dos columnas, y realicen lo siguiente:
a. Verificar si las columnas se pueden combinar en una reducción suficiente
b. Generar la matriz del canal determinante necesario para combinar las columnas """
def matriz_ceros(matriz, filas, columnas):
  for i in range(filas):
    fila = []
    for j in range(columnas):
      fila.append(0)
    matriz.append(fila)
  return matriz
def se_puede_combinar(matriz_canal, col1, col2):
  cant_filas = len(matriz_canal)

  r = -1
  for i in range(cant_filas):
    if matriz_canal[i][col2] != 0 and r == -1:
      r = i

  if r == -1:
    return True

  for i in range(cant_filas):
    izquierda = matriz_canal[i][col1] * matriz_canal[r][col2]
    derecha = matriz_canal[r][col1] * matriz_canal[i][col2]
    if abs(izquierda - derecha) > 0.000001:
      return False
  return True
def genera_matriz_determinante(matriz_canal, col1, col2):
  if not se_puede_combinar(matriz_canal, col1, col2):
    return False
  else:
    matriz_determinante=[]
    cant_columnas_original=len(matriz_canal[0])
    cant_columnas=cant_columnas_original-1
    matriz_determinante=matriz_ceros(matriz_determinante, cant_columnas_original, cant_columnas)
    menor = min(col1, col2)
    mayor = max(col1, col2)
    for j in range(cant_columnas_original):
      if j < mayor:
        destino = j
      elif j == mayor:
        destino = menor
      else:
        destino = j - 1
      matriz_determinante[j][destino] = 1
    return matriz_determinante
def main():
  canal = [[0.5, 0.25, 0.25], [0.2, 0.4, 0.4]]
  print("Caso 1, columnas 1 y 2:")
  print("Se puede combinar:", se_puede_combinar(canal, 1, 2))
  resultado = genera_matriz_determinante(canal, 1, 2)
  for fila in resultado:
    print(fila)

  canal = [[0.2, 0.4, 0.4], [0.1, 0.2, 0.7]]
  print("Caso 2, columnas 0 y 1:")
  print("Se puede combinar:", se_puede_combinar(canal, 0, 1))
  resultado = genera_matriz_determinante(canal, 0, 1)
  for fila in resultado:
    print(fila)

  canal = [[0.5, 0.5, 0.0], [0.0, 0.5, 0.5]]
  print("Caso 3, columnas 0 y 1:")
  print("Se puede combinar:", se_puede_combinar(canal, 0, 1))
  print("Determinante:", genera_matriz_determinante(canal, 0, 1))

main()


Caso 1, columnas 1 y 2:
Se puede combinar: True
[1, 0]
[0, 1]
[0, 1]
Caso 2, columnas 0 y 1:
Se puede combinar: True
[1, 0]
[1, 0]
[0, 1]
Caso 3, columnas 0 y 1:
Se puede combinar: False
Determinante: False


In [ ]:
"""23. Codificar una función en Python que reciba como parámetro la matriz de un canal y, utilizando las funciones de los ejercicios 20 y 22,
realice todas las reducciones suficientes posibles y devuelva la matriz del canal reducido. """
def matriz_ceros(matriz, filas, columnas):
  for i in range(filas):
    fila = []
    for j in range(columnas):
      fila.append(0)
    matriz.append(fila)
  return matriz
def crea_matriz_compuesta(matriz_canal1, matriz_canal2):
  cant_filas=len(matriz_canal1)
  cant_columnas=len(matriz_canal2[0])
  dimension=len(matriz_canal2)
  matriz_compuesta = []
  matriz_compuesta = matriz_ceros(matriz_compuesta, cant_filas, cant_columnas)
  for i in range(cant_filas):
    for k in range(cant_columnas):
      for j in range(dimension):
        matriz_compuesta[i][k] = matriz_compuesta[i][k] + matriz_canal1[i][j] * matriz_canal2[j][k]
  return matriz_compuesta
def se_puede_combinar(matriz_canal, col1, col2):
  cant_filas = len(matriz_canal)

  r = -1
  for i in range(cant_filas):
    if matriz_canal[i][col2] != 0 and r == -1:
      r = i

  if r == -1:
    return True

  for i in range(cant_filas):
    izquierda = matriz_canal[i][col1] * matriz_canal[r][col2]
    derecha = matriz_canal[r][col1] * matriz_canal[i][col2]
    if abs(izquierda - derecha) > 0.000001:
      return False
  return True
def genera_matriz_determinante(matriz_canal, col1, col2):
  if not se_puede_combinar(matriz_canal, col1, col2):
    return False
  else:
    matriz_determinante=[]
    cant_columnas_original=len(matriz_canal[0])
    cant_columnas=cant_columnas_original-1
    matriz_determinante=matriz_ceros(matriz_determinante, cant_columnas_original, cant_columnas)
    menor = min(col1, col2)
    mayor = max(col1, col2)
    for j in range(cant_columnas_original):
      if j < mayor:
        destino = j
      elif j == mayor:
        destino = menor
      else:
        destino = j - 1
      matriz_determinante[j][destino] = 1
    return matriz_determinante
def busca_par_combinable(matriz_canal):
  cant_columnas = len(matriz_canal[0])
  for col1 in range(cant_columnas):
    for col2 in range(col1 + 1, cant_columnas):
      if se_puede_combinar(matriz_canal, col1, col2):
        return col1, col2
  return -1, -1

def reduce_canal(matriz_canal):
  while True:
    col1, col2 = busca_par_combinable(matriz_canal)
    if col1 == -1:
      break
    determinante = genera_matriz_determinante(matriz_canal, col1, col2)
    matriz_canal = crea_matriz_compuesta(matriz_canal, determinante)
  return matriz_canal
def main():
  canal = [[0.2, 0.4, 0.4], [0.1, 0.2, 0.7]]
  print("Ejemplo 1:")
  for fila in reduce_canal(canal):
    print(fila)

  canal = [[0.1, 0.1, 0.4, 0.4], [0.3, 0.3, 0.2, 0.2]]
  print("Ejemplo 2:")
  for fila in reduce_canal(canal):
    print(fila)

  canal = [[0.5, 0.5, 0.0], [0.0, 0.5, 0.5]]
  print("Ejemplo 3:")
  for fila in reduce_canal(canal):
    print(fila)

main()

Ejemplo 1:
[0.6000000000000001, 0.4]
[0.30000000000000004, 0.7]
Ejemplo 2:
[0.2, 0.8]
[0.6, 0.4]
Ejemplo 3:
[0.5, 0.5, 0.0]
[0.0, 0.5, 0.5]


In [ ]:
"""25. Implementar funciones en Python que reciban como parámetro la matriz de un canal y resuelvan lo siguiente:
a. Verificar si se trata de un canal uniforme
b. Calcular su capacidad en los siguientes casos:
i. Canal determinante
ii. Canal sin ruido
iii. Canal uniforme """
import math
def canal_sin_ruido(matriz_canal):
  cant_filas = len(matriz_canal)
  cant_columnas = len(matriz_canal[0])
  for j in range(cant_columnas):
    cont=0
    for i in range(cant_filas):
      if matriz_canal[i][j]!=0:
        cont+=1
    if cont!=1:
      return False
  return True
def canal_determinante(matriz_canal):
  cant_filas = len(matriz_canal)
  cant_columnas = len(matriz_canal[0])
  for i in range(cant_filas):
    cont=0
    for j in range(cant_columnas):
      if matriz_canal[i][j]!=0:
        cont+=1
    if cont!=1:
      return False
  return True
def verifica_canal_uniforme(matriz_canal):
  primera_fila = sorted(matriz_canal[0])
  cant_filas = len(matriz_canal)
  for i in range(cant_filas):
    if sorted(matriz_canal[i]) != primera_fila:
      return False
  return True
def calcular_capacidad(matriz_canal):
  if canal_determinante(matriz_canal):
    return math.log2(len(matriz_canal[0]))
  elif canal_sin_ruido(matriz_canal):
    return math.log2(len(matriz_canal))
  elif verifica_canal_uniforme(matriz_canal):
    cant_columnas = len(matriz_canal[0])
    entropia = 0
    for j in range(cant_columnas):
      if matriz_canal[0][j] > 0:
        entropia = entropia + matriz_canal[0][j] * math.log2(1 / matriz_canal[0][j])
    return math.log2(cant_columnas)-entropia
  else:
    return -1
def main():
  matrices = [
    [[1, 0], [0, 1], [0, 1]],
    [[0.5, 0.5, 0], [0, 0, 1]],
    [[0.9, 0.1], [0.1, 0.9]],
    [[0.5, 0.25, 0.25], [0.25, 0.5, 0.25], [0.25, 0.25, 0.5]],
    [[0.5, 0.5], [0.2, 0.8]]
  ]

  for matriz in matrices:
    print("Matriz:", matriz)
    print("  Uniforme:", verifica_canal_uniforme(matriz))
    print("  Capacidad:", calcular_capacidad(matriz))

main()


Matriz: [[1, 0], [0, 1], [0, 1]]
  Uniforme: True
  Capacidad: 1.0
Matriz: [[0.5, 0.5, 0], [0, 0, 1]]
  Uniforme: False
  Capacidad: 1.0
Matriz: [[0.9, 0.1], [0.1, 0.9]]
  Uniforme: True
  Capacidad: 0.5310044064107187
Matriz: [[0.5, 0.25, 0.25], [0.25, 0.5, 0.25], [0.25, 0.25, 0.5]]
  Uniforme: True
  Capacidad: 0.08496250072115608
Matriz: [[0.5, 0.5], [0.2, 0.8]]
  Uniforme: False
  Capacidad: -1


In [ ]:
"""26. Realizar una función en Python que reciba como parámetros: la matriz de un canal binario y un valor de paso, y estime la capacidad del canal
 mediante el cálculo de la información mutua para un conjunto de probabilidades a priori distribuidas uniformemente según el paso especificado.
 La función debe retornar el valor de capacidad estimado, junto con su probabilidad asociada. """
def matriz_ceros(matriz):
  cant_filas = len(matriz)
  cant_columnas = len(matriz[0])
  matriz_salida = []
  for i in range(cant_filas):
    fila = []
    for j in range(cant_columnas):
      fila.append(0)
    matriz_salida.append(fila)
  return matriz_salida

def crea_matriz_eventos_simultaneos(matriz_canal,prob_priori):
  matriz_eventos_simultaneos = matriz_ceros(matriz_canal)
  cant_filas = len(matriz_canal)
  cant_columnas = len(matriz_canal[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      matriz_eventos_simultaneos[i][j] = matriz_canal[i][j] * prob_priori[i]
  return matriz_eventos_simultaneos

def crea_matriz_posteriori(matriz_eventos_simultaneos,prob_simbolos_salida):
  matriz_posteriori = matriz_ceros(matriz_eventos_simultaneos)
  cant_filas = len(matriz_eventos_simultaneos)
  cant_columnas = len(matriz_eventos_simultaneos[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      matriz_posteriori[i][j] = matriz_eventos_simultaneos[i][j] / prob_simbolos_salida[j]
  return matriz_posteriori

def crea_prob_simbolos_salida(matriz_eventos_simultaneos):
  cant_columnas=len(matriz_eventos_simultaneos[0])
  cant_filas=len(matriz_eventos_simultaneos)
  prob_simbolos_salida = []
  for j in range(cant_columnas):
    prob_simbolos_salida.append(0)
  for j in range(cant_columnas):
    for i in range(cant_filas):
      prob_simbolos_salida[j]+=matriz_eventos_simultaneos[i][j]
  return prob_simbolos_salida
def calcula_valores(prob_priori, matriz_canal):
  simultaneos = crea_matriz_eventos_simultaneos(matriz_canal, prob_priori)
  prob_salida = crea_prob_simbolos_salida(simultaneos)
  posteriori = crea_matriz_posteriori(simultaneos, prob_salida)
  return posteriori, prob_salida, simultaneos
def calcular_informacion_mutua(prob_priori, matriz_canal):
  posteriori, prob_salida, simultaneos = calcula_valores(prob_priori, matriz_canal)
  acumulado=0
  cant_filas = len(posteriori)
  cant_columnas = len(posteriori[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      if simultaneos[i][j] > 0:
        acumulado = acumulado + simultaneos[i][j] * math.log2(simultaneos[i][j] / (prob_priori[i] * prob_salida[j]))
  return acumulado
def estimativo_capacidad_canal(matriz_canal_binario, valor_paso):
  cant_pasos = round(1/valor_paso)
  capacidad = -1
  prob_capacidad = 0
  for k in range (1, cant_pasos):
    p = k * valor_paso
    prob_priori = [p,1-p]
    inf_mutua = calcular_informacion_mutua(prob_priori, matriz_canal_binario)
    if inf_mutua > capacidad:
      capacidad = inf_mutua
      prob_capacidad = p
  return capacidad, prob_capacidad
def main():
  canal = [[0.9, 0.1], [0.1, 0.9]]
  capacidad, prob = estimativo_capacidad_canal(canal, 0.1)
  print("Canal simétrico, paso 0.1:")
  print("  Capacidad estimada:", capacidad)
  print("  Probabilidad asociada:", prob)

  capacidad, prob = estimativo_capacidad_canal(canal, 0.25)
  print("Canal simétrico, paso 0.25:")
  print("  Capacidad estimada:", capacidad)
  print("  Probabilidad asociada:", prob)

  canal = [[1, 0], [0.5, 0.5]]
  capacidad, prob = estimativo_capacidad_canal(canal, 0.1)
  print("Canal [[1, 0], [0.5, 0.5]], paso 0.1:")
  print("  Capacidad estimada:", capacidad)
  print("  Probabilidad asociada:", prob)

main()

Canal simétrico, paso 0.1:
  Capacidad estimada: 0.5310044064107189
  Probabilidad asociada: 0.5
Canal simétrico, paso 0.25:
  Capacidad estimada: 0.5310044064107189
  Probabilidad asociada: 0.5
Canal [[1, 0], [0.5, 0.5]], paso 0.1:
  Capacidad estimada: 0.32192809488736246
  Probabilidad asociada: 0.6000000000000001


In [ ]:
"""30. Desarrollar una función en Python que reciba como parámetros: una lista con probabilidades a priori y la matriz de un canal,
y calcule la probabilidad de error utilizando la regla de decisión de máxima posibilidad."""
def matriz_ceros(matriz):
  cant_filas = len(matriz)
  cant_columnas = len(matriz[0])
  matriz_salida = []
  for i in range(cant_filas):
    fila = []
    for j in range(cant_columnas):
      fila.append(0)
    matriz_salida.append(fila)
  return matriz_salida

def crea_matriz_eventos_simultaneos(matriz_canal,prob_priori):
  matriz_eventos_simultaneos = matriz_ceros(matriz_canal)
  cant_filas = len(matriz_canal)
  cant_columnas = len(matriz_canal[0])
  for i in range(cant_filas):
    for j in range(cant_columnas):
      matriz_eventos_simultaneos[i][j] = matriz_canal[i][j] * prob_priori[i]
  return matriz_eventos_simultaneos
def calcula_prob_error(prob_priori, matriz_canal):
  cant_columnas=len(matriz_canal[0])
  cant_filas=len(matriz_canal)
  i = 0
  matriz_eventos_simultaneos = crea_matriz_eventos_simultaneos(matriz_canal, prob_priori)
  prob_aciertos = 0
  for j in range(cant_columnas):
    maximo = -1
    fila_max = 0
    for i in range(cant_filas):
      if matriz_canal[i][j] > maximo:
        maximo = matriz_canal[i][j]
        fila_max = i
    prob_aciertos = prob_aciertos + matriz_eventos_simultaneos[fila_max][j]
  return 1 - prob_aciertos
def main():
  casos = [
    ([0.5, 0.5], [[0.9, 0.1], [0.1, 0.9]]),
    ([0.5, 0.5], [[1, 0], [0.5, 0.5]]),
    ([0.9, 0.1], [[0.6, 0.4], [0.3, 0.7]]),
    ([0.5, 0.5], [[0.5, 0.5], [0.5, 0.5]])
  ]

  for prob_priori, matriz_canal in casos:
    print("A priori:", prob_priori, "Canal:", matriz_canal)
    print("  Probabilidad de error:", calcula_prob_error(prob_priori, matriz_canal))

main()

A priori: [0.5, 0.5] Canal: [[0.9, 0.1], [0.1, 0.9]]
  Probabilidad de error: 0.09999999999999998
A priori: [0.5, 0.5] Canal: [[1, 0], [0.5, 0.5]]
  Probabilidad de error: 0.25
A priori: [0.9, 0.1] Canal: [[0.6, 0.4], [0.3, 0.7]]
  Probabilidad de error: 0.39
A priori: [0.5, 0.5] Canal: [[0.5, 0.5], [0.5, 0.5]]
  Probabilidad de error: 0.5
